In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import math
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pickle
%matplotlib inline
from IPython.display import clear_output
from typing import List, Tuple, Optional

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# simulations_folder = "drive/MyDrive/mestrado_final_files/simulations/NGramModel_ExtraInfo_two_models/"
# models_token_folder = "drive/MyDrive/mestrado_final_files/models/NGramModel_ExtraInfo/"
# models_time_folder = "drive/MyDrive/mestrado_final_files/models/SpentTimeModel/"

data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
simulations_folder = "G:\Meu Drive\mestrado_final_files\simulations\\NGramModel_ExtraInfo_two_models\\"
models_token_folder = "G:\Meu Drive\mestrado_final_files\models\\NGramModel_ExtraInfo\\"
models_time_folder = "G:\Meu Drive\mestrado_final_files\models\\SpentTimeModel\\"


In [3]:
print(f"Pandas version: {pd.__version__},\nNumpy Version: {np.__version__},\
      \nTorch Version: {torch.__version__},\nMatplotlib Version: {plt.matplotlib.__version__}")

Pandas version: 2.2.3,
Numpy Version: 2.2.6,      
Torch Version: 2.8.0+cpu,
Matplotlib Version: 3.10.0


In [4]:
pd.set_option("display.expand_frame_repr", False)

In [5]:
tokens_info = pd.read_csv(data_folder + 'tokens_info.txt', sep = ';')\
.sort_values(['cod', 'token'])\
.set_index('token')

# Convert the index of tokens_info to a list of tokens
list_tokens = tokens_info.index.to_list()

# Create a dictionary mapping each token to a unique integer, starting from 1
token_to_integer = {tok: idx + 1 for idx, tok in enumerate(list_tokens)}

# Assign a special integer (0) for the period (.)
token_to_integer['.'] = 0

# Create an inverse mapping from integers back to tokens
integer_to_token = {idx: tok for tok, idx in token_to_integer.items()}

# tokens_info = tokens_info\
# .query("hpts != 0 or vpts != 0 or limhteam != 0 or limvteam != 0")\
# .to_dict(orient = 'index')


In [6]:
%run 02_functions_simulations.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/02_functions_simulations.ipynb

In [7]:
with open(data_folder + 'dict_set_covariables.pkl', 'rb') as f:
    dict_set_covariables = pickle.load(f)

In [8]:
with open(data_folder + 'prop_end_period_by_remaining_time.pkl', 'rb') as f:
    dict_prop_end_period_remaining_time = pickle.load(f)

In [9]:
list_models = [item for item in os.listdir(models_token_folder) if "NGramModel_ExtraInfo_first46m_seasondata" in item and ".pth" in item]

In [10]:
VOCAB_SIZE = 601
TIME_SIZE = 301

In [11]:
# i = 0
# for model_filename1 in list_models:
#     i += 1
#     print(i)

#     parquet_filename = ("simulations_" + model_filename1.replace("_parameters.pth", "").replace("first46m", "segregated") + ".parquet")
#     time_model_filename1 = model_filename1.replace("NGramModel_ExtraInfo", "SpentTimeModel")

#     model_filename2 = model_filename1.replace("first46m", "last2m")
#     time_model_filename2 = time_model_filename1.replace("first46m", "last2m")

#     if parquet_filename in os.listdir(simulations_folder):
#         print("already simulated")
#         continue

#     season_ = model_filename1.split("_seasondata")[1].split("_")[0]
#     CONTEXT_SIZE = int(model_filename1.split("_context")[1].split("_")[0])
#     EMBEDDING_LAYER_SIZE = int(model_filename1.split("_embedding")[1].split("_")[0])
#     variables_config = int(model_filename1.split("_setcovariables")[1].split("_")[0])
#     numerical_input_size = len(dict_set_covariables['index'][variables_config])
#     num_hidden_neurons = int(model_filename1.split("_numneurons")[1].split("_")[0])
#     num_layers = int(model_filename1.split("_numlayers")[1].split("_")[0])
#     dropout_rate = 0 if "nodropout" in model_filename1 else int(model_filename1.split("_dropout")[1].split("_")[0]) / 10
#     layer_norm = True if model_filename1.split("_layernorm")[1].split("_")[0] == 'True' else False

#     token_model1 = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
#                                         numerical_input_dim = numerical_input_size, num_hidden_neurons = num_hidden_neurons,
#                                         num_layers = num_layers, activation = nn.GELU, dropout_rate = dropout_rate,
#                                         use_layer_norm = layer_norm)

#     token_model1.load_state_dict(torch.load(models_token_folder + model_filename1, weights_only = True, map_location = torch.device('cpu')))
#     token_model1.eval()

#     time_model1 = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
#                                  embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = numerical_input_size,
#                                  num_hidden_neurons = num_hidden_neurons, num_layers = num_layers,
#                                  activation_function = nn.GELU, dropout_rate = dropout_rate, use_layer_norm = layer_norm)


#     time_model1.load_state_dict(torch.load(models_time_folder + time_model_filename1, weights_only = True, map_location = torch.device('cpu')))
#     time_model1.eval()

#     token_model2 = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
#                                         numerical_input_dim = numerical_input_size, num_hidden_neurons = num_hidden_neurons,
#                                         num_layers = num_layers, activation = nn.GELU, dropout_rate = dropout_rate,
#                                         use_layer_norm = layer_norm)

#     token_model2.load_state_dict(torch.load(models_token_folder + model_filename2, weights_only = True, map_location = torch.device('cpu')))
#     token_model2.eval()

#     time_model2 = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
#                                  embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = numerical_input_size,
#                                  num_hidden_neurons = num_hidden_neurons, num_layers = num_layers,
#                                  activation_function = nn.GELU, dropout_rate = dropout_rate, use_layer_norm = layer_norm)

#     time_model2.load_state_dict(torch.load(models_time_folder + time_model_filename2, weights_only = True, map_location = torch.device('cpu')))
#     time_model2.eval()

#     # df_simulations = generate_game_plays_2models(token_model1 = token_model1, token_model2 = token_model2, time_model1 = time_model1,
#     #                                              time_model2 = time_model2, num_simulations = 10000,
#     #                                              token_context = CONTEXT_SIZE, num_numerical_variables = numerical_input_size,
#     #                                              config = variables_config)

#     df_simulations = generate_game_plays_v2(class_model = "ExtraInfo", token_model1 = token_model1, time_model1 = time_model1,
#                                             token_model2 = token_model2, time_model2 = time_model2, num_simulations = 1000,
#                                             token_context = CONTEXT_SIZE, num_numerical_variables = numerical_input_size,
#                                             config_setvariables = variables_config,
#                                             matrix_end_period_probs = dict_prop_end_period_remaining_time[season_])

#     df_simulations.to_parquet(simulations_folder + parquet_filename, index = False, compression = "zstd")
#     clear_output(wait = True)

In [12]:
i = 0
for model_filename1 in [item for item in list_models if "_context5_embedding32" in item]:
    i += 1
    print(i)

    time_model_filename1 = model_filename1.replace("NGramModel_ExtraInfo", "SpentTimeModel")

    model_filename2 = model_filename1.replace("first46m", "last2m")
    time_model_filename2 = time_model_filename1.replace("first46m", "last2m")

    # if parquet_filename in os.listdir(simulations_folder):
    #     print("already simulated")
    #     continue

    season_ = model_filename1.split("_seasondata")[1].split("_")[0]
    CONTEXT_SIZE = int(model_filename1.split("_context")[1].split("_")[0])
    EMBEDDING_LAYER_SIZE = int(model_filename1.split("_embedding")[1].split("_")[0])
    variables_config = int(model_filename1.split("_setcovariables")[1].split("_")[0])
    numerical_input_size = len(dict_set_covariables['index'][variables_config])
    num_hidden_neurons = int(model_filename1.split("_numneurons")[1].split("_")[0])
    num_layers = int(model_filename1.split("_numlayers")[1].split("_")[0])
    dropout_rate = 0 if "nodropout" in model_filename1 else int(model_filename1.split("_dropout")[1].split("_")[0]) / 10
    layer_norm = True if model_filename1.split("_layernorm")[1].split("_")[0] == 'True' else False

    token_model1 = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                        numerical_input_dim = numerical_input_size, num_hidden_neurons = num_hidden_neurons,
                                        num_layers = num_layers, activation = nn.GELU, dropout_rate = dropout_rate,
                                        use_layer_norm = layer_norm)

    token_model1.load_state_dict(torch.load(models_token_folder + model_filename1, weights_only = True, map_location = torch.device('cpu')))
    token_model1.eval()

    time_model1 = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                                 embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = numerical_input_size,
                                 num_hidden_neurons = num_hidden_neurons, num_layers = num_layers,
                                 activation_function = nn.GELU, dropout_rate = dropout_rate, use_layer_norm = layer_norm)


    time_model1.load_state_dict(torch.load(models_time_folder + time_model_filename1, weights_only = True, map_location = torch.device('cpu')))
    time_model1.eval()

    token_model2 = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                        numerical_input_dim = numerical_input_size, num_hidden_neurons = num_hidden_neurons,
                                        num_layers = num_layers, activation = nn.GELU, dropout_rate = dropout_rate,
                                        use_layer_norm = layer_norm)

    token_model2.load_state_dict(torch.load(models_token_folder + model_filename2, weights_only = True, map_location = torch.device('cpu')))
    token_model2.eval()

    time_model2 = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                                 embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = numerical_input_size,
                                 num_hidden_neurons = num_hidden_neurons, num_layers = num_layers,
                                 activation_function = nn.GELU, dropout_rate = dropout_rate, use_layer_norm = layer_norm)

    time_model2.load_state_dict(torch.load(models_time_folder + time_model_filename2, weights_only = True, map_location = torch.device('cpu')))
    time_model2.eval()

    # df_simulations = generate_game_plays_2models(token_model1 = token_model1, token_model2 = token_model2, time_model1 = time_model1,
    #                                              time_model2 = time_model2, num_simulations = 10000,
    #                                              token_context = CONTEXT_SIZE, num_numerical_variables = numerical_input_size,
    #                                              config = variables_config)

    df_simulations = generate_game_plays_v2(class_model = "ExtraInfo", token_model1 = token_model1, time_model1 = time_model1,
                                            token_model2 = token_model2, time_model2 = time_model2, num_simulations = 10000,
                                            token_context = CONTEXT_SIZE, num_numerical_variables = numerical_input_size,
                                            config_setvariables = variables_config,
                                            matrix_end_period_probs = dict_prop_end_period_remaining_time[season_])

    for parte in range(0, 10):
        parquet_filename = ("simulations_" + model_filename1.replace("_parameters.pth", "").replace("first46m", "segregated") + "_parte" + str(parte + 1) + ".parquet")

        df_simulations\
        .assign(parte = lambda df: (df["sim_id"] // 1000))\
        .query("parte == @parte")\
        .to_parquet(simulations_folder + parquet_filename, index = False, compression = "zstd")

        #df_simulations.to_parquet(simulations_folder + parquet_filename, index = False, compression = "zstd")
    del df_simulations
    clear_output(wait = True)

Period: 6
keep_generating vector distribution: {0: 10000}
                  count      mean        std   min    1%    5%   10%    25%    50%    75%    90%    95%    99%    max
home_points       10000  114.1351  11.801934  74.0  88.0  95.0  99.0  106.0  114.0  122.0  129.0  134.0  143.0  158.0
away_points       10000  111.4621  11.568758  70.0  86.0  93.0  97.0  104.0  111.0  119.0  126.0  131.0  139.0  153.0
perc_played_time  10000    0.1300   3.603207   0.0   0.0   0.0   0.0    0.0    0.0    0.0    0.0    0.0    0.0  100.0
Result: {np.str_('away'): 4379, np.str_('home'): 5621}
